# 03 — Neo4j: modelagem e investigação
> **LAB-VET-01 — estudo com PaySim (dados sintéticos).** Não use dados confidenciais, credenciais ou identificadores reais neste repositório. Execute as células em ordem. Os resultados históricos estão documentados separadamente; as saídas foram removidas para publicação.

**Modelo:** `(Account)-[:SENT]->(Transaction)-[:RECEIVED_BY]->(Account)`. `RECEIVED_BY` reflete o campo `nameDest`, não comprova liquidação financeira. As consultas completas estão em `cypher/`.

**Atenção:** a ingestão é incremental e usa `MERGE`. Para reproduzir as contagens, execute em um banco vazio e dedicado.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Coloque o CSV público PaySim em data/ (não versionar arquivos de dados).
DATA_PATH = Path("data/PS_20174392719_1491204439457_log.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset ausente: {DATA_PATH.resolve()}. Consulte o README.")
df = pd.read_csv(DATA_PATH)
print("Dataset:", df.shape)

In [ ]:
treino = df.loc[df["step"] <= 300].copy()
validacao = df.loc[(df["step"] > 300) & (df["step"] <= 400)].copy()
teste = df.loc[df["step"] > 400].copy()
print("Treino:", treino.shape, "Validação:", validacao.shape, "Teste:", teste.shape)

In [ ]:
import pandas as pd

# Começaremos com transações do período de treino
# para validar a modelagem sem utilizar dados futuros.

df_grafo = treino.loc[
    treino["step"] <= 100,
    [
        "step",
        "type",
        "amount",
        "nameOrig",
        "nameDest",
        "isFraud"
    ]
].copy()

# Identificador único baseado no índice original
df_grafo["tx_id"] = (
    "TX_" + df_grafo.index.astype(str)
)

# Organizar as colunas
df_grafo = df_grafo[
    [
        "tx_id",
        "step",
        "type",
        "amount",
        "nameOrig",
        "nameDest",
        "isFraud"
    ]
]

print("Transações preparadas:", len(df_grafo))
print("IDs únicos:", df_grafo["tx_id"].nunique())

display(df_grafo.head())

In [ ]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os
load_dotenv()
config = [os.getenv("NEO4J_URI"), os.getenv("NEO4J_USER"), os.getenv("NEO4J_PASSWORD")]
if not all(config):
    raise ValueError("Configure NEO4J_URI, NEO4J_USER e NEO4J_PASSWORD no .env local (não versionar).")
driver = GraphDatabase.driver(config[0], auth=(config[1], config[2]))
driver.verify_connectivity()
print("Conexão confirmada")

In [ ]:
query_constraint = """
CREATE CONSTRAINT account_id_unique IF NOT EXISTS
FOR (a:Account)
REQUIRE a.account_id IS UNIQUE
"""

with driver.session() as session:
    session.run(query_constraint).consume()

print("Constraint de Account criada ou já existente!")

In [ ]:
query_tx_constraint = """
CREATE CONSTRAINT transaction_id_unique IF NOT EXISTS
FOR (t:Transaction)
REQUIRE t.tx_id IS UNIQUE
"""

with driver.session() as session:
    session.run(query_tx_constraint).consume()

print("Constraint de Transaction criada ou já existente!")

## Ingestão parametrizada em lotes — até 100 mil

In [ ]:
lote_teste = df_grafo.head(100).to_dict("records")

query_ingestao = """
UNWIND $transacoes AS tx

MERGE (origem:Account {
    account_id: tx.nameOrig
})

MERGE (destino:Account {
    account_id: tx.nameDest
})

MERGE (t:Transaction {
    tx_id: tx.tx_id
})

SET t.step = toInteger(tx.step),
    t.type = tx.type,
    t.amount = toFloat(tx.amount)

MERGE (origem)-[:SENT]->(t)
MERGE (t)-[:RECEIVED_BY]->(destino)
"""

with driver.session() as session:
    session.run(
        query_ingestao,
        transacoes=lote_teste
    ).consume()

print("100 transações processadas com sucesso!")

In [ ]:
from time import perf_counter
inicio = perf_counter()
with driver.session() as session:
    for inicio_lote in range(0, 100_000, 1000):
        lote = df_grafo.iloc[inicio_lote:inicio_lote+1000].to_dict("records")
        session.run(query_ingestao, transacoes=lote).consume()
print("Tempo (s):", round(perf_counter()-inicio, 2))

In [ ]:
query_verificacao = """
MATCH (t:Transaction)
RETURN count(t) AS total_transacoes
"""

with driver.session() as session:
    resultado = session.run(query_verificacao).single()

    print(
        "Transações no Neo4j:",
        resultado["total_transacoes"]
    )

In [ ]:
query_auditoria = """
MATCH (t:Transaction)

OPTIONAL MATCH (origem:Account)-[:SENT]->(t)
OPTIONAL MATCH (t)-[:RECEIVED_BY]->(destino:Account)

RETURN
    count(DISTINCT t) AS transacoes,
    count(DISTINCT origem) AS origens_unicas,
    count(DISTINCT destino) AS destinos_unicos,
    count(DISTINCT CASE
        WHEN origem IS NULL OR destino IS NULL
        THEN t
    END) AS transacoes_incompletas
"""

with driver.session() as session:
    resultado = session.run(query_auditoria).single()
    print(dict(resultado))

## Auditoria retrospectiva de rótulos — apenas para análise, nunca para features

In [ ]:
conta = "C985934102"

amostra_neo4j = df_grafo.head(100_000)

transacoes_amostra = amostra_neo4j[
    amostra_neo4j["nameDest"] == conta
]

transacoes_completas = df_grafo[
    df_grafo["nameDest"] == conta
]

print(
    "Primeiras 100 mil:",
    len(transacoes_amostra)
)

print(
    "Todo df_grafo:",
    len(transacoes_completas)
)

print(
    "Diferença:",
    len(transacoes_completas)
    - len(transacoes_amostra)
)

print(
    "\nFraudes na amostra:",
    transacoes_amostra["isFraud"].sum()
)

In [ ]:
amostra = df_grafo.head(100_000)

print("Transações:", len(amostra))

print(
    "Fraudes:",
    int(amostra["isFraud"].sum())
)

print(
    "Taxa de fraude:",
    f'{amostra["isFraud"].mean():.4%}'
)

print("\nFraudes por tipo:")

print(
    amostra[amostra["isFraud"] == 1]
    .groupby("type")
    .size()
)

**Ao terminar:** feche a conexão.

In [ ]:
driver.close()

---
**Autoria**

```text
Alexandre Lotti
<!LOTTI>
```
